# 1.6 如何用參數列示預測偏好？


[1.5](https://birdhackor.github.io/tiny-perceptron-vlm/1.5.html) 的表靠次數決定機率。現在想換一種方式：不手動計數，而是讓每格存一個可調分數，之後依據答案好壞逐步調整。像評審先為候選人記分，再依得分決定推薦比例，分數可以為正、為負，也不必加總為 1。我們把這些可調的數字叫參數（parameter）；學習將改它們，文字 ID 的對照則仍固定。

用三個字「貓、看、狗」建立三列三欄的分數表。第 0 列代表目前字是貓，第 1 列是看，第 2 列是狗；每欄分別評下一字為貓、看、狗的偏好。若第 0 列是 `[0,2,0]`，表示看到貓時比較偏好看。它還不是機率；下一節會把分數轉成合法比例。本節只核對查表是否取對了列。

PyTorch 的 `nn.Embedding` 是可學習的查表工具，Embedding 常譯為嵌入層。`nn.Embedding(3,3)` 建立三列，每列三個數。這裡故意直接拿三個數當三個候選字的分數；第 2 章會用同一工具取得少量特徵，再交給後續計算，功能相同、用途不同。tensor 與軸見 [W.3](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

scores = nn.Embedding(3, 3)
with torch.no_grad():
    scores.weight.zero_()
    scores.weight[0, 1] = 2.0
inputs = torch.tensor([0, 2])
output = scores(inputs)
print(scores.weight)
print(output)

輸入 `[0,2]` 要求分別查「目前是貓」與「目前是狗」兩列。`scores.weight` 是層儲存的那張參數列，`zero_()` 把每格設為 0，再手動把第 0 列第 1 欄設為 2。結尾底線提醒這個方法直接修改原物件。`torch.no_grad()` 表示這次手動設定不用記錄求導關係，後面的學習會在 [1.9](https://birdhackor.github.io/tiny-perceptron-vlm/1.9.html) 開始介紹求導。

第一個輸出顯示整張表 `[[0,2,0],[0,0,0],[0,0,0]]`，可能附帶 `Parameter` 等標記，那是在提醒你這是可調參數。第二個輸出只有 `[[0,2,0],[0,0,0]]`，形狀 `[2,3]`：兩個輸入各取一列，每列仍有三個候選分數。第 1 列因為沒有被輸入查詢，就不在第二個輸出裡。這裡尚未執行任何更新，數字都是我們指定的。

若字表有 V 個字，這種方法需要 V×V 個參數；每個目前字都有一整排下一字候選分數。比如一百個字便有一萬格。這是一個很小的語言模型：根據輸入產生候選分數、數字可以改進，但目前仍只看前一字，無法知道更遠的內容。ID 之間的距離沒有拿來計算，查第 0 列或第 2 列只是選不同的規則。

練習在設定區加入 `scores.weight[2, 0] = 3.0`。執行前預測查表結果 `output` 的第0列是否改變；核對 `output[0]` 應仍是 `[0,2,0]`，`output[1]` 變成 `[3,0,0]`。你只調了狗那列，不會影響貓那列。這讓「參數在哪裡」和「哪個輸入用到它」變成可以直接追蹤的事。
